# **Reading the Image Dataset**

In [ ]:
import cv2
import os
import numpy as np
import pandas as pd
from math import exp, sqrt
import matplotlib.pyplot as plt
from sklearn import svm, datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.utils.multiclass import unique_labels
import matplotlib.image as mpimg
from sklearn.utils import shuffle
import keras
import tensorflow as tf
import tensorflow.keras.backend as K
from keras import Input
from keras.models import Model
from keras.layers import Dense, Dropout, Activation, Flatten
from keras.layers import  BatchNormalization, Concatenate, Add, DepthwiseConv2D, GlobalAveragePooling2D, SeparableConv2D
from tensorflow.keras.optimizers import SGD,RMSprop,Adam,Adagrad
from keras.layers import Conv2D, MaxPooling2D

In [ ]:
data_path = '/workspace/notebooks/Mri - Alzimers/Data /NewMRI'
data_dir_list = os.listdir(data_path)
img_data_list = []
labels = []

for dataset in data_dir_list:
    img_list = os.listdir(os.path.join(data_path, dataset))
    print('Loaded the images of dataset-{}\n'.format(dataset))
    for img in img_list:
        if img.lower().endswith(('.png', '.jpg', '.jpeg')):
            input_img = cv2.imread(os.path.join(data_path, dataset, img))
            if input_img is not None:
                img = cv2.resize(input_img, (224, 224))
                labels.append(dataset)
                img_data_list.append(img)

label = np.array(labels)
img_data = np.array(img_data_list)
img_data = img_data.astype('float32')
img_data = img_data / 255.0
print(img_data.shape)

Loaded the images of dataset-CN

Loaded the images of dataset-SMC

Loaded the images of dataset-.ipynb_checkpoints

Loaded the images of dataset-LMCI

Loaded the images of dataset-MCI

Loaded the images of dataset-EMCI

Loaded the images of dataset-AD

(9742, 224, 224, 3)


# **Splitting the Data into Training and Testing**

In [ ]:
from tensorflow.keras.utils import to_categorical
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

class_names = ['CN', 'SMC', 'LMCI', 'MCI', 'EMCI', 'AD']
class_mapping = {name: idx for idx, name in enumerate(class_names)}
integer_labels = [class_mapping[label] for label in labels]
num_classes = len(class_names)
Y = to_categorical(integer_labels, num_classes)
x, y = shuffle(img_data, Y, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (7793, 224, 224, 3)
X_test shape: (1949, 224, 224, 3)
y_train shape: (7793, 6)
y_test shape: (1949, 6)


In [ ]:
X_train = X_train.reshape(-1, 224, 224, 3)
X_test = X_test.reshape(-1, 224, 224, 3)

In [ ]:
import keras
from keras.models import Model
from tensorflow.keras import regularizers
from keras.layers import Input, Conv2D, Add, Multiply, Concatenate, AveragePooling2D, Flatten, DepthwiseConv2D, BatchNormalization
from keras.layers import SeparableConv2D, MaxPooling2D, Dense, GlobalAveragePooling2D, Activation
from tensorflow.keras.applications import ResNet50, VGG16, MobileNet

base_model = MobileNet(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_model.layers[:12]:
    layer.trainable = False
x = base_model.output
x1 = Conv2D(64, (3, 3),  dilation_rate = 1, padding = 'same', kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01), activation='relu')(x)
x1 = BatchNormalization()(x1)
x2 = Conv2D(64, (3, 3),  dilation_rate = 3, padding = 'same', kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01), activation='relu')(x)
x2 = BatchNormalization()(x2)
x3 = Conv2D(64, (3, 3),  dilation_rate = 5, padding = 'same', kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01), activation='relu')(x)
x3 = BatchNormalization()(x3)

A1 = Add()([x1, x2])
M1 = Multiply()([x2, x3])

x4 = Conv2D(64, (3, 3), padding = 'same', kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01), activation='relu')(A1)
x4 = BatchNormalization()(x4)
g1 = GlobalAveragePooling2D()(x4)
d1 = Dense(1, activation = 'sigmoid')(g1)
M3 = Multiply()([d1, x1])

x5 = Activation('softmax')(M1)
M2 = Multiply()([x3, x5])

C1 = Concatenate()([M2, M3])
x5 = Conv2D(64, (3, 3), padding = 'same', kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01), activation='relu')(C1)
C2 = Concatenate()([x5, x])

x = GlobalAveragePooling2D()(C2)
x = Dense(128, activation='relu')(x)
x = Dense(64, activation='relu')(x)
x = Dense(32, activation='relu')(x)
output = Dense(6, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['acc'])
model.summary()

2024-08-13 10:05:12.972357: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1926] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 17947 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB MIG 3g.20gb, pci bus id: 0000:b7:00.0, compute capability: 8.0


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 conv1 (Conv2D)              (None, 112, 112, 32)         864       ['input_1[0][0]']             
                                                                                                  
 conv1_bn (BatchNormalizati  (None, 112, 112, 32)         128       ['conv1[0][0]']               
 on)                                                                                              
                                                                                                  
 conv1_relu (ReLU)           (None, 112, 112, 32)         0         ['conv1_bn[0][0]']        

In [ ]:
from keras.callbacks import EarlyStopping, ModelCheckpoint
early_stopping = EarlyStopping(
                              patience=30,
                              min_delta=0.001,
                              monitor="val_acc",
                              restore_best_weights=True
                              )
# Define the model checkpoint callback to save the best weights
checkpoint = ModelCheckpoint('/workspace/notebooks/Mri - Alzimers/Data /Check/'+'-{epoch:02d}.h5', monitor='val_acc', save_best_only=True)


history = model.fit( X_train, y_train, batch_size = 32, epochs = 200,
                    validation_data = (X_test , y_test), callbacks=[early_stopping, checkpoint], verbose =1)

Epoch 1/200


2024-08-13 10:05:26.084894: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:467] Loaded cuDNN version 90100
2024-08-13 10:05:27.414811: I external/local_xla/xla/service/service.cc:168] XLA service 0x7fcd422022a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-08-13 10:05:27.414849: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA A100-SXM4-40GB MIG 3g.20gb, Compute Capability 8.0
2024-08-13 10:05:27.421370: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1723543527.545970  675577 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


244/244 [==============================] - ETA: 0s - loss: 16.6323 - acc: 0.4985

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


244/244 [==============================] - 41s 90ms/step - loss: 16.6323 - acc: 0.4985 - val_loss: 4.1946 - val_acc: 0.4397
Epoch 2/200
244/244 [==============================] - 13s 53ms/step - loss: 3.3145 - acc: 0.6095 - val_loss: 6.1572 - val_acc: 0.2329
Epoch 3/200
244/244 [==============================] - 13s 54ms/step - loss: 3.0736 - acc: 0.6996 - val_loss: 4.2449 - val_acc: 0.3853
Epoch 4/200
244/244 [==============================] - 13s 52ms/step - loss: 2.8311 - acc: 0.7881 - val_loss: 4.3581 - val_acc: 0.5064
Epoch 5/200
244/244 [==============================] - 13s 55ms/step - loss: 2.6952 - acc: 0.8427 - val_loss: 3.4537 - val_acc: 0.6491
Epoch 6/200
244/244 [==============================] - 13s 53ms/step - loss: 2.5651 - acc: 0.8904 - val_loss: 3.9673 - val_acc: 0.5208
Epoch 7/200
244/244 [==============================] - 12s 49ms/step - loss: 2.5240 - acc: 0.9080 - val_loss: 4.5589 - val_acc: 0.4808
Epoch 8/200
244/244 [==============================] - 12s 48ms/st

In [ ]:
model_size = model.count_params() * 4 / (1024 * 1024)
print("Model Size (MB):", model_size)

In [ ]:
import keras
model = keras.models.load_model('/workspace/Ostro/Data/check/-54.h5')

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

# Predicting the Test set results
pred = model.predict(X_test)
print("Y_pred:", pred)
print("*****************")
y_pred = np.argmax(pred, axis = 1)
y_true = np.argmax(y_test, axis = 1)

In [ ]:
#Making the Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
target_names = ['Normal', 'Osteoporosis']
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
sns.set(font_scale=1.25)
p = sns.heatmap(pd.DataFrame(cm), annot=True,xticklabels=target_names, yticklabels=target_names, cmap="YlGnBu" ,fmt='g')
# plt.title('Confusion matrix', y=1.1)
plt.ylabel('Actual label')
plt.xlabel('Predicted label')
plt.savefig('/workspace/Ostro/Data/CM.png')

In [ ]:
#Making the Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
target_names = ['Normal', 'Osteoprosis']
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
sns.set(font_scale=1.25)
cm = cm.astype('float')/cm.sum(axis=1)[:, np.newaxis]
p = sns.heatmap(pd.DataFrame(cm), annot=True,xticklabels=target_names, yticklabels=target_names, cmap="YlGnBu" ,fmt='.3f')
plt.ylabel('Actual label')
plt.xlabel('Predicted label')
plt.savefig('/workspace/Ostro/Data/CM/CM2.png')

In [ ]:
#import classification_report
from sklearn.metrics import classification_report
print(classification_report(y_true,y_pred, target_names = target_names))

In [ ]:
from sklearn.metrics import roc_curve, auc
# sns.set(font_scale=1.25)
y_pred_proba = model.predict(X_test)
fpr, tpr, thresholds = roc_curve(y_true, y_pred)
roc_auc = auc(fpr, tpr)
plt.plot([0,1],[0,1],'k--')
plt.plot(fpr,tpr, color='green', label='AUC (area = %0.2f)' % roc_auc)
plt.xlabel('False Positive Rate', fontsize = 15)
plt.ylabel('True Positive Rate', fontsize = 15)
plt.legend(loc="lower right")
plt.title('ROC curve')
plt.grid(True)
plt.axis(True)
plt.savefig('/workspace/Ostro/Data/CM/ROC2.png')
plt.show()